In [1]:
from __future__ import annotations


In [2]:
%reload_ext autoreload
%autoreload 3

In [3]:

import numpy as np

from minitorch.tensor.tensor import Tensor
from minitorch.activations.activations import GELU
from minitorch.nn.layers import Linear, Flatten, LayerNormalization, BatchNormalization
from minitorch.attention.attention import MultiHeadAttention
from minitorch.embendding.embed import EmbeddingLayer


def create_causal_mask(seq_len: int) -> Tensor:
    """
    Create a causal mask (autoregressive mask).
    
    This create the causal mask to make sure that tokens i
    only communicates to token j where j<i.
    Essential for autoregressive GPT models.

    Args:
        seq_len (int): Length of the sequence

    Returns:
        Tensor: Tensor of shape (1, seq_len, seq_len) with:
        - 1.0 for positions that CAN be attended to (lower triangle)
        - 0.0 for positions that CANNOT be attended to (upper triangle)
    """
    mask = np.tril(np.ones(shape=(seq_len, seq_len), dtype= np.float32))
    return Tensor(mask[np.newaxis, :, :])

In [4]:
arr = Tensor.rand(high=1, low=0, shape=(5,6), requires_grad=True)

In [5]:
arr

Tensor(data=[[0.83703744 0.4422989  0.79242826 0.04658349 0.85594076 0.5455652 ]
 [0.35470274 0.83547884 0.50731313 0.5359518  0.0487684  0.10663801]
 [0.20110653 0.00931657 0.70376277 0.25533864 0.90491825 0.11584476]
 [0.19485945 0.3802711  0.90630746 0.7012361  0.45242164 0.25399378]
 [0.11889769 0.46478286 0.9341134  0.7123214  0.12893301 0.8560692 ]], shape=(5, 6), grad_info= True)

In [13]:
norm = LayerNormalization(arr.shape[1])
mask_norm = norm(arr)

In [14]:
mask_norm.backward()

In [15]:
norm.weight.grad, norm.bias.grad

(array([0., 0., 0., 0., 0., 0.], dtype=float32),
 array([0., 0., 0., 0., 0., 0.], dtype=float32))

In [ ]:
EPSILON = 1e-9

class LayerNormalization:
    """
    Normalizes across the last dimension of the input tensor
    """
    def __init__(self, num_features: int, eps: float = EPSILON) -> None:
        if not isinstance(num_features, int) or num_features <= 0:
            raise ValueError(
                f"num_features must be a positive integer, got {num_features}"
            )

        self.dim = num_features
        self.eps = eps

        # Learnable parameters
        self.weight = Tensor(np.ones(shape=self.dim))
        self.bias = Tensor(np.zeros(shape=self.dim))

    def forward(self, X: Tensor) -> Tensor:
        assert X.shape[-1] == self.dim, \
            f'last dim of input should equal dim, received: {X.shape[-1]} expected: {self.dim}'

        mean = X.mean(axis=-1, keepdims=True)
        var = X.var(axis=-1, keepdims=True)
        std = (var + self.eps) ** 0.5

        normalized = (X - mean) / std
        shifted_norm = self.weight * normalized + self.bias

        def _backward():
            if not shifted_norm.requires_grad:
                return

            grad_output = shifted_norm.grad  # upstream gradient

            # ---- Bias gradient ----
            if self.bias.requires_grad:
                grad_bias = grad_output
                while grad_bias.ndim > self.bias.data.ndim:
                    grad_bias = grad_bias.sum(axis=0)
                self.bias.grad = (
                    grad_bias if self.bias.grad is None else self.bias.grad + grad_bias
                )

            # ---- Weight gradient ----
            if self.weight.requires_grad:
                grad_weight = grad_output * normalized
                while grad_weight.ndim > self.weight.data.ndim:
                    grad_weight = grad_weight.sum(axis=0)
                self.weight.grad = (
                    grad_weight if self.weight.grad is None else self.weight.grad + grad_weight
                )

            # ---- Input gradient ----
            if X.requires_grad:
                N = self.dim

                grad_norm = grad_output * self.weight  # dL/d(normalized)

                mean_grad = np.mean(grad_norm, axis=-1, keepdims=True)
                mean_grad_norm = np.mean(grad_norm * normalized, axis=-1, keepdims=True)

                grad_x = (1.0 / std.data) * (
                    grad_norm - mean_grad - normalized.data * mean_grad_norm
                )

                X.grad = grad_x if X.grad is None else X.grad + grad_x

        shifted_norm._backward = _backward
        shifted_norm.requires_grad = (
            X.requires_grad or self.weight.requires_grad or self.bias.requires_grad
        )

        return shifted_norm

In [ ]:
norm = LayerNormalization(arr.shape[1])

In [ ]:
arr_norm = norm.forward(arr)

In [ ]:
norm.weight.grad, norm.bias.grad

(array([0., 0., 0., 0., 0., 0.], dtype=float32),
 array([0., 0., 0., 0., 0., 0.], dtype=float32))

In [ ]:
arr_norm.backward()

In [ ]:
arr.grad

array([[Tensor(data=-3.3616061045904644e-07, shape=(), grad_info= True),
        Tensor(data=4.6569317646572017e-07, shape=(), grad_info= True),
        Tensor(data=-4.934071853313071e-07, shape=(), grad_info= True),
        Tensor(data=1.3604626758478844e-07, shape=(), grad_info= True),
        Tensor(data=3.218217443645699e-07, shape=(), grad_info= True),
        Tensor(data=-9.399346367899852e-08, shape=(), grad_info= True)],
       [Tensor(data=-6.580388145493998e-08, shape=(), grad_info= True),
        Tensor(data=4.3813071215481614e-07, shape=(), grad_info= True),
        Tensor(data=1.4614775523114076e-07, shape=(), grad_info= True),
        Tensor(data=-1.2038815100368083e-07, shape=(), grad_info= True),
        Tensor(data=-5.18303806984477e-07, shape=(), grad_info= True),
        Tensor(data=1.2021735074085882e-07, shape=(), grad_info= True)],
       [Tensor(data=6.71708164645679e-08, shape=(), grad_info= True),
        Tensor(data=-1.8919169519904244e-07, shape=(), grad_info